In [ ]:
# !rm -rf data
# !rm -rf prediction
# !rm tunnel_url.txt

In [ ]:
!pip install terratorch -t /kaggle/temp/custom_libs -qqq

In [ ]:
!pip install whitebox planetary-computer pystac-client rioxarray -t /kaggle/temp/custom_libs -qqq

In [ ]:
import sys
sys.path.insert(0,"/kaggle/temp/custom_libs")

In [ ]:
import os
import sys
import glob
import time
import zipfile
import warnings
import subprocess

import torch
import numpy as np
import pandas as pd
import albumentations as A
import lightning.pytorch as pl
import rasterio
import terratorch.datamodules

from pathlib import Path
from albumentations.pytorch import ToTensorV2
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from terratorch.tasks import SemanticSegmentationTask
from lightning.pytorch.callbacks import LearningRateMonitor

# from litlogger import LightningLogger
warnings.filterwarnings('ignore')
import cv2
import os
import glob
import subprocess
import zipfile
import warnings
import numpy as np
import rasterio
import cv2
# import earthaccess
from pathlib import Path
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling

warnings.filterwarnings('ignore')
from torch.utils.data import Dataset

import rasterio
import os
from glob import glob

import os
import numpy as np
import rasterio
from glob import glob

import os
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.merge import merge
from pystac_client import Client
import planetary_computer as pc
from glob import glob
import pyproj
from scipy.ndimage import generic_filter

In [ ]:
# ==========================================
# 1. CONFIGURATION (DeepSARFlood Compliant)
# ==========================================
class Config:
    # Use Absolute paths to avoid "0 images found" issues in nested Kaggle/Local setups
    BASE_DIR = Path(".").resolve()
    DATA_ROOT = BASE_DIR / "data"
    
    # Competition Data (Nested structure handling)
    COMP_DATA = DATA_ROOT
    TRAIN_IMG_DIR = COMP_DATA / "image"
    TRAIN_LBL_DIR = COMP_DATA / "label"
    PRED_IMG_DIR = COMP_DATA / "prediction" / "image"
    
    # Outputs (10-Band Optimized Stacks)
    TRAIN_10BAND = DATA_ROOT / "image_10band"
    PRED_10BAND = DATA_ROOT / "prediction_10band"

    # DeepSARFlood Feature Order:
    # [HH, HV, LogDiff, Ratio, DEM, Slope, Green, Red, NIR, SWIR]

In [ ]:
# ==========================================
# 2. CORE COMPONENTS
# ==========================================

def download_competition_data(comp_name):
    """Ensures raw data is present."""
    if not Config.COMP_DATA.exists():
        print(f"📥 Downloading competition: {comp_name}")
        subprocess.run(["kaggle", "competitions", "download", "-c", comp_name], check=True)
        zip_file = f"{comp_name}.zip"
        if os.path.exists(zip_file):
            with zipfile.ZipFile(zip_file, 'r') as z:
                z.extractall(Config.BASE_DIR)
            os.remove(zip_file)

In [ ]:
def isolate_swir_band(input_base_path):
    """
    Keep HH, HV, SWIR. Fill others with zeros.
    
    Final band order (6 bands):
    Band 1 → HH
    Band 2 → HV
    Band 3 → SWIR  ✅
    Band 4 → 0
    Band 5 → 0
    Band 6 → 0
    """

    image_paths = glob(os.path.join(input_base_path, "**/image/*.tif"), recursive=True)
    print(f"Found {len(image_paths)} images to process...")

    for img_path in image_paths:
        with rasterio.open(img_path) as src:
            meta = src.meta.copy()

            # --- READ REQUIRED BANDS ---
            hh = src.read(1)   # adjust if needed
            hv = src.read(2)   # adjust if needed
            swir = src.read(6) # your SWIR

            # --- CREATE OUTPUT STACK ---
            output_data = np.zeros((6, src.height, src.width), dtype='float32')

            output_data[0] = hh
            output_data[1] = hv
            output_data[2] = swir  # SWIR at 3rd position

            # --- UPDATE META ---
            meta.update({
                "count": 6,
                "dtype": 'float32',
                "nodata": 0
            })

        # --- WRITE BACK ---
        with rasterio.open(img_path, 'w', **meta) as dst:
            dst.write(output_data)

    print("✅ Done: HH, HV kept. SWIR moved to Band 3. Others zero-filled.")


# Usage
# isolate_swir_band("data")

In [ ]:
def get_master_dem(image_paths, master_out="master_dem.tif"):
    if os.path.exists(master_out): return master_out
    print("[1/2] Fetching Master DEM for the whole area...")
    
    lons, lats = [], []
    for path in image_paths:
        with rasterio.open(path) as src:
            transformer = pyproj.Transformer.from_crs(src.crs, "epsg:4326", always_xy=True)
            w, s, e, n = src.bounds
            p1_lon, p1_lat = transformer.transform(w, s)
            p2_lon, p2_lat = transformer.transform(e, n)
            lons.extend([p1_lon, p2_lon]); lats.extend([p1_lat, p2_lat])

    bbox = [min(lons)-0.05, min(lats)-0.05, max(lons)+0.05, max(lats)+0.05]
    catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=pc.sign_inplace)
    items = catalog.search(collections=["cop-dem-glo-30"], bbox=bbox).item_collection()
    
    src_files = [rasterio.open(i.assets["data"].href) for i in items]
    mosaic, out_trans = merge(src_files)
    out_meta = src_files[0].meta.copy()
    out_meta.update({"height": mosaic.shape[1], "width": mosaic.shape[2], "transform": out_trans})
    
    with rasterio.open(master_out, "w", **out_meta) as dest:
        dest.write(mosaic)
    return master_out

def fast_topo_pipeline(data_dir):
    image_paths = glob(os.path.join(data_dir, "**/image/*.tif"), recursive=True)
    master_dem_path = get_master_dem(image_paths)

    print(f"[2/2] Processing {len(image_paths)} patches locally...")
    with rasterio.open(master_dem_path) as master:
        for path in image_paths:
            filename = os.path.basename(path)
            with rasterio.open(path) as src:
                meta = src.meta.copy()
                dem = np.zeros((meta['height'], meta['width']), dtype='float32')
                reproject(rasterio.band(master, 1), dem, src_transform=master.transform, 
                          src_crs=master.crs, dst_transform=meta['transform'], 
                          dst_crs=meta['crs'], resampling=Resampling.bilinear)

            # --- Fast Slope (Band 6) ---
            # Gradient method is instant
            dy, dx = np.gradient(dem, 18.0)
            slope = np.arctan(np.sqrt(dx**2 + dy**2)) * (180 / np.pi)

            # --- Fast HAND Proxy (Band 5) ---
            # Instead of D8 flow, we use 'Elevation above local minimum'
            # This identifies low-lying basins which are flood-prone
            local_min = generic_filter(dem, np.min, size=25) # ~450m window
            hand_proxy = dem - local_min

            # --- Write to TIF ---
            with rasterio.open(path, 'r+') as dst:
                dst.write(dem.astype('float32'), 4)         # Band 4: DEM
                dst.write(hand_proxy.astype('float32'), 5)  # Band 5: Fast-HAND
                dst.write(slope.astype('float32'), 6)       # Band 6: Slope
            
            print(f"[SUCCESS] {filename}")

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds
from pystac_client import Client
import planetary_computer as pc
from glob import glob
import pyproj


def is_north_up(transform):
    # North-up rasters usually have negative y pixel size
    return transform.e < 0


def get_master_dry_sar_v5(image_paths, master_out="master_dry_reference.tif"):
    """
    Build a dry SAR reference from a single usable dry tile.
    This avoids merge() completely and fixes upside-down rasters by flipping
    the array only when needed.
    """
    if os.path.exists(master_out):
        try:
            with rasterio.open(master_out) as ds:
                if is_north_up(ds.transform):
                    print(f"[INFO] {master_out} already exists and is north-up.")
                    return master_out
        except Exception:
            pass

    if not image_paths:
        raise Exception("No input images found.")

    # 1) Calculate bbox from local flood images
    lons, lats = [], []
    for path in image_paths:
        with rasterio.open(path) as src:
            transformer = pyproj.Transformer.from_crs(src.crs, "epsg:4326", always_xy=True)
            w, s, e, n = src.bounds
            p1_lon, p1_lat = transformer.transform(w, s)
            p2_lon, p2_lat = transformer.transform(e, n)
            lons.extend([p1_lon, p2_lon])
            lats.extend([p1_lat, p2_lat])

    bbox = [min(lons) - 0.05, min(lats) - 0.05, max(lons) + 0.05, max(lats) + 0.05]

    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace
    )

    # 2) Try date/polarization combinations until one works
    search_configs = [
        {"datetime": "2024-01-01/2024-03-15", "pol": "HH", "label": "2024 HH"},
        {"datetime": "2023-01-01/2023-03-15", "pol": "HH", "label": "2023 HH"},
        {"datetime": "2024-01-01/2024-03-15", "pol": "VV", "label": "2024 VV"},
        {"datetime": "2023-01-01/2023-03-15", "pol": "VV", "label": "2023 VV"},
        {"datetime": "2022-01-01/2022-03-15", "pol": "HH", "label": "2022 HH"},
        {"datetime": "2022-01-01/2022-03-15", "pol": "VV", "label": "2022 VV"},
    ]

    chosen_item = None
    pol_key = None

    for config in search_configs:
        search = catalog.search(
            collections=["sentinel-1-grd"],
            bbox=bbox,
            datetime=config["datetime"]
        )
        items = search.item_collection()

        valid_items = [
            item for item in items
            if config["pol"] in item.properties.get("sar:polarizations", [])
        ]

        if valid_items:
            chosen_item = valid_items[0]
            pol_key = config["pol"].lower()
            print(f"[FOUND] Using {config['label']} for dry reference.")
            break

    if chosen_item is None:
        raise Exception("No dry SAR data found for any fallback date.")

    # 3) Read ONE tile only, no merge
    href = chosen_item.assets[pol_key].href
    with rasterio.open(href) as src:
        data = src.read(1)

        # Fix upside-down raster by flipping the array, not by merging/reprojecting
        if src.transform.e > 0:
            data = np.flipud(data)
            transform = from_bounds(*src.bounds, src.width, src.height)
        else:
            transform = src.transform

        meta = src.meta.copy()
        meta.update(
            count=1,
            dtype="float32",
            transform=transform
        )

        with rasterio.open(master_out, "w", **meta) as dst:
            dst.write(data.astype("float32"), 1)

    print(f"[SUCCESS] {master_out} created.")
    return master_out


def run_final_band_pipeline(data_dir):
    image_paths = glob(os.path.join(data_dir, "**/image/*.tif"), recursive=True)
    master_dry_path = get_master_dry_sar_v5(image_paths)

    print(f"Processing Band 7 injection for {len(image_paths)} images...")

    with rasterio.open(master_dry_path) as master:
        for path in image_paths:
            filename = os.path.basename(path)

            with rasterio.open(path) as src:
                if src.count >= 7:
                    print(f"[SKIP] {filename} already has 7 bands.")
                    continue

                if src.count < 6:
                    print(f"[SKIP] {filename} has fewer than 6 bands.")
                    continue

                meta = src.meta.copy()

                # Read only the first 6 bands
                data_6_bands = src.read(indexes=[1, 2, 3, 4, 5, 6]).astype("float32")
                current_sar = data_6_bands[0]  # Band 1 HH

                # Align dry reference to this image
                dry_ref = np.zeros((meta["height"], meta["width"]), dtype="float32")
                reproject(
                    rasterio.band(master, 1),
                    dry_ref,
                    src_transform=master.transform,
                    src_crs=master.crs,
                    dst_transform=meta["transform"],
                    dst_crs=meta["crs"],
                    resampling=Resampling.bilinear
                )

            # Band 7 = flood SAR - dry SAR
            diff_band = current_sar - dry_ref

            # Write all 7 bands as float32 to preserve the difference band
            meta.update(count=7, dtype="float32")
            temp_path = path + ".tmp"

            with rasterio.open(temp_path, "w", **meta) as dst:
                for i in range(6):
                    dst.write(data_6_bands[i].astype("float32"), i + 1)
                dst.write(diff_band.astype("float32"), 7)

            os.replace(temp_path, path)
            print(f"[SUCCESS] Band 7 added: {filename}")


# if __name__ == "__main__":
#     run_final_band_pipeline("data")

In [ ]:
def data_pipeline():
    
    # Download Data and Extract
    download_competition_data("anrfaisehack-theme-1-phase2")
    
    # removes all bands except SWIR and Move that to 3 rd band
    isolate_swir_band("data")

    # Adds topographic features (DEM, Slope) and Fast-HAND to create 10-band stacks
    fast_topo_pipeline("data")

In [ ]:
data_pipeline()

In [ ]:
# ==========================================
# 1. CONFIGURATION (Single Source of Truth)
# ==========================================
class Config:
    DATA_ROOT = "./data"
    OUTPUT_DIR = "./Output"
    CHECKPOINT_DIR = "./Output/checkpoints"
    PRED_INPUT = "./data/prediction/image"
    PRED_OUTPUT = "./prediction"
    SUBMISSION_CSV = "./submission.csv"

    # Dataset Specs
    NUM_CLASSES = 3
    # Means/Stds for HH, HV, Green, Red, NIR, SWIR
    # how to verify if 0 is a band and 7 is not ! 
    BANDS = [1, 2, 3, 4, 5, 6]
    IMAGE_PATTERN = "*image.tif"
    LABEL_PATTERN = "*label.tif"

    MEANS = [801.78878784, 357.10271464, 1275.30988244, 3.32332926, 1.92494704, 1.43010689]
    STDS  = [435.30253145, 164.03019541, 543.94006356, 2.01572047, 1.86861661, 1.49314347]

    # Hardware / DataLoader
    BATCH_SIZE = 6
    NUM_WORKERS = 0
    PIN_MEMORY = True
    PERSISTENT_WORKERS = False

    # Hyperparameters
    LR = 2e-5
    WEIGHT_DECAY = 0.01
    MAX_EPOCHS = 90
    ACCUMULATE_GRAD = 4
    PRECISION = "16-mixed"
    SEED = 42

    # Architecture
    BACKBONE = "prithvi_eo_v2_100_tl"
    DECODER = "UperNetDecoder"
    DECODER_CHANNELS = 512
    DROPOUT = 0.4
    FREEZE_BACKBONE = False

    CLASS_WEIGHTS = [1.0, 5.0, 3.0]

In [ ]:
# ==========================================
# 3. UTILITIES: RLE & SUBMISSION
# ==========================================
# Convert prediction tif files to Kaggle-style run-length encoding (RLE) Submission csv
def mask_to_rle(mask):
    """
    Convert binary mask to RLE (Kaggle format).
    Mask must be 2D numpy array with values 0 or 1.
    """
    pixels = mask.flatten(order="F")  # column-major
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

def generate_submission(tif_dir, output_csv):
    """
    Read TIF predictions and generate Kaggle submission CSV.
    Logic kept identical to competition standards.
    """
    tif_dir = Path(tif_dir)
    rows = []

    for tif_path in sorted(tif_dir.glob("*.tif")):
        with rasterio.open(tif_path) as src:
            mask = src.read(1)

        # UPDATE THIS LINE: Only target the specific class the competition wants
        # Change the '1' to a '2' if class 2 is the actual flood class!
        TARGET_CLASS = 1 
        mask = (mask == TARGET_CLASS).astype(np.uint8)

        rle = mask_to_rle(mask)

        rows.append({
            "id": tif_path.name.replace("_image.tif", ""),
            "rle_mask": rle
        })

    df = pd.DataFrame(rows)
    df = df.replace("", 0).fillna(0) # replace null/ na with zero - kaggle compatible
    df.to_csv(output_csv, index=False)
    print(f"Saved Kaggle RLE CSV : {output_csv}")

In [ ]:
def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),

        ToTensorV2(),

    ])

In [ ]:
# ==========================================
# 4. DATAMODULE & MODEL BUILDERS
# ==========================================
def build_datamodule(cfg):
    return terratorch.datamodules.GenericNonGeoSegmentationDataModule(
        batch_size=cfg.BATCH_SIZE,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
        persistent_workers=cfg.PERSISTENT_WORKERS,
        num_classes=cfg.NUM_CLASSES,
        train_data_root=f"{cfg.DATA_ROOT}/image",
        train_label_data_root=f"{cfg.DATA_ROOT}/label",
        val_data_root=f"{cfg.DATA_ROOT}/image",
        val_label_data_root=f"{cfg.DATA_ROOT}/label",
        test_data_root=f"{cfg.DATA_ROOT}/image",
        test_label_data_root=f"{cfg.DATA_ROOT}/label",
        train_split=f"{cfg.DATA_ROOT}/split/train.txt",
        val_split=f"{cfg.DATA_ROOT}/split/val.txt",
        test_split=f"{cfg.DATA_ROOT}/split/test.txt",
        img_grep=cfg.IMAGE_PATTERN,
        label_grep=cfg.LABEL_PATTERN,
        train_transform=get_train_transforms(),
        means=cfg.MEANS,
        stds=cfg.STDS,
        no_data_replace=0,
        no_label_replace=-1,
        predict_data_root=cfg.PRED_INPUT
    )

def build_model(cfg):
    model_args = {
        "backbone": cfg.BACKBONE,
        "backbone_pretrained": True,
        "backbone_bands": cfg.BANDS,
        "backbone_num_frames": 1,
        "decoder": cfg.DECODER,
        "decoder_channels": cfg.DECODER_CHANNELS,
        "decoder_scale_modules": True,
        "num_classes": cfg.NUM_CLASSES,
        "head_dropout": cfg.DROPOUT,
        "rescale": True,
        # Match the helper code's simplified neck for V2 models
        "necks": [
        dict(
            name="ReshapeTokensToImage",
            effective_time_dim=1,
        )
    ]
}

    return SemanticSegmentationTask(
        model_args=model_args,
        plot_on_val=False,
        class_weights=cfg.CLASS_WEIGHTS,
        loss = {"ce": 0.4, "dice": 0.6},
        lr=cfg.LR,
        optimizer="AdamW",
        optimizer_hparams={"weight_decay": cfg.WEIGHT_DECAY},
        ignore_index=-1,
        freeze_backbone=cfg.FREEZE_BACKBONE,
        model_factory="EncoderDecoderFactory",
        scheduler="ReduceLROnPlateau",
        # scheduler_hparams={"T_mult": 2},
    )

def get_loggers(cfg):
    loggers = [
        TensorBoardLogger(save_dir=cfg.OUTPUT_DIR, name="tb_logs"),
        CSVLogger(save_dir=cfg.OUTPUT_DIR, name="csv_logs")
    ]
    return loggers

In [ ]:
import torch.nn.functional as F

# ==========================================
# 6. SUBMISSION GENERATOR (Phase 2 Optimized)
# ==========================================
def generate_phase2_submission(tif_dir, output_csv):
    """
    Specifically targets Class 1 (Flood) for RLE.
    Handles '0 0' requirement for empty masks.
    """
    tif_dir = Path(tif_dir)
    rows = []

    for tif_path in sorted(tif_dir.glob("*.tif")):
        with rasterio.open(tif_path) as src:
            mask = src.read(1)

        # PHASE 2 CRITICAL: Target ONLY Class 1 (Flood)
        # 0: No Flood, 1: Flood, 2: Water Body
        binary_flood_mask = (mask == 1).astype(np.uint8)

        # mask_to_rle should be defined globally as per your previous cells
        rle = mask_to_rle(binary_flood_mask)
        
        # HACKATHON RULE: If empty, must be "0 0"
        if not rle or rle.strip() == "":
            rle = "0 0"

        rows.append({
            "id": tif_path.name.replace("_image.tif", ""),
            "rle_mask": rle
        })

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"🚀 FINAL SUBMISSION READY: {output_csv}")

# ==========================================
# 5. MAIN EXECUTION PIPELINE
# ==========================================
def main():

    cfg = Config()
    pl.seed_everything(cfg.SEED)
    for d in [cfg.CHECKPOINT_DIR, cfg.PRED_OUTPUT]: os.makedirs(d, exist_ok=True)
    torch.set_float32_matmul_precision("high")

    # 3. Initialize Model and Trainer
    dm = build_datamodule(cfg)
    model = build_model(cfg)

    checkpoint_cb = ModelCheckpoint(
        monitor="val/IoU_1",   # ✅ FIXED (correct key)
        mode="max", 
        dirpath=cfg.CHECKPOINT_DIR,
        filename="best-flood-{epoch:02d}-{val/IoU_1:.4f}",  # ✅ also fix here
        save_top_k=1,
    )
    
    trainer = pl.Trainer(
        accelerator="gpu", devices=1, precision=cfg.PRECISION,
        max_epochs=cfg.MAX_EPOCHS, accumulate_grad_batches=cfg.ACCUMULATE_GRAD,
        logger=get_loggers(cfg), 
        callbacks=[checkpoint_cb, LearningRateMonitor(logging_interval='step')],
        num_sanity_val_steps=2, log_every_n_steps=5
    )

    # 4. Training Phase
    print(">>> Starting Training...")
    trainer.fit(model, datamodule=dm)

    # 5. Validation/Test with Best Checkpoint
    best_path = checkpoint_cb.best_model_path
    if best_path:
        print(f">>> Testing with best checkpoint: {best_path}")
        trainer.test(model, datamodule=dm, ckpt_path=best_path)

    # 6. Prediction & Uncertainty Logic (DeepSARFlood Approach)
    print(">>> Generating Predictions and Confidence Maps...")
    model.eval()
    dm.setup("predict")

    UNCERTAINTY_DIR = os.path.join(cfg.OUTPUT_DIR, "uncertainty_maps")
    os.makedirs(UNCERTAINTY_DIR, exist_ok=True)

    # Note: Use best_path to ensure we are predicting with the top-performing model
    predictions = trainer.predict(model, datamodule=dm, ckpt_path=best_path if best_path else None)

    for batch_idx, (logits, file_paths) in enumerate(predictions):
        if isinstance(logits, tuple): logits = logits[0]
        
        # Softmax to get probabilities for all 3 classes
        preds_class = logits.cpu().numpy().astype("int16")
        
        uncertainty = np.zeros_like(preds_class, dtype="float32")

        for i in range(preds_class.shape[0]):
            ref_path = file_paths[i]
            base_name = os.path.basename(ref_path)
            arr = preds_class[i]
            unc_arr = uncertainty[i]

            # Handle no-data values correctly
            arr[arr < 0] = -1
            
            with rasterio.open(ref_path) as src:
                meta = src.meta.copy()

            # --- Save the 3-Class Segmentation Result ---
            meta.update({"count": 1, "dtype": "int16", "nodata": -1, "compress": "lzw"})
            out_path = os.path.join(cfg.PRED_OUTPUT, base_name)
            with rasterio.open(out_path, "w", **meta) as dst:
                dst.write(arr, 1)

            # --- Save the Uncertainty Map ---
            meta.update({"dtype": "float32", "nodata": -1})
            unc_out_path = os.path.join(UNCERTAINTY_DIR, base_name.replace(".tif", "_uncertainty.tif"))
            with rasterio.open(unc_out_path, "w", **meta) as dst:
                dst.write(unc_arr, 1)

    # 7. Final Submission CSV
    generate_phase2_submission(cfg.PRED_OUTPUT, cfg.SUBMISSION_CSV)
    print(">>> Pipeline Finished Successfully!")

In [ ]:
import os
import urllib.request
import time

# 1. Start TensorBoard in the background
print("🚀 Starting TensorBoard in the background...")
os.system("tensorboard --logdir ./Output --host 0.0.0.0 --port 6005 &")

# 2. Install localtunnel silently
print("📦 Installing localtunnel...")
os.system("npm install -g localtunnel -qqq")

# 3. Fetch the Kaggle machine's external IP
endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print("="*50)
print(f"🛑 IMPORTANT: Copy this IP address: {endpoint_ip}")
print("You will need to paste this as the 'Endpoint IP' on the warning page.")
print("="*50)

# 4. Start the reverse tunnel IN THE BACKGROUND
# Using 'nohup' and '&' completely detaches it from the cell execution
print("🔗 Creating tunnel in the background...")
os.system("nohup lt --port 6005 > tunnel_url.txt 2>&1 &")

# Give the tunnel 4 seconds to connect to the server and generate the link
time.sleep(4)

# 5. Read the generated URL from the text file and print it
print("✅ Tunnel is running! Here is your link:")
with open("tunnel_url.txt", "r") as f:
    print(f.read().strip())

print("\n🎉 Done! This cell is now free and you can run other cells.")

In [ ]:
if __name__ == "__main__":
    # 2. Data Preparation

    import gc
    import cv2
    cv2.setNumThreads(0)
    gc.collect()
    torch.cuda.empty_cache()
    main()